# Raahi ML Full Training

This notebook documents training and evaluation flows for the five trainable Raahi datasets:

1. `raw_transport_dataset.csv`
2. `Raahi_Behavioral_Chakachakit.csv`
3. `Raahi_Mumbai_Route_Transport_Noisy.csv`
4. `Raahi_Full_Western_Line_Fares.csv`
5. `vehicle_emission_dataset.csv`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

from raahi_ml.config.ml_config import DATA_RAW_PATH, RANDOM_STATE, TEST_SIZE
from raahi_ml.pipelines.preprocess_new import preprocess_behavioral, preprocess_route, preprocess_transport

results = {}


## 1. Transport Mode Classifier

Dataset: `raw_transport_dataset.csv`  
Task: multiclass classification for `mode`.

In [ ]:
transport_df = pd.read_csv(DATA_RAW_PATH / 'raw_transport_dataset.csv')
X_transport, y_transport, transport_encoders = preprocess_transport(transport_df)
X_train, X_test, y_train, y_test = train_test_split(
    X_transport, y_transport, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_transport
)
transport_model = XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, eval_metric='mlogloss')
transport_model.fit(X_train, y_train)
transport_pred = transport_model.predict(X_test)
results['transport_mode_classifier_accuracy'] = accuracy_score(y_test, transport_pred)
ConfusionMatrixDisplay.from_predictions(y_test, transport_pred)
plt.title('Transport Mode Classifier Confusion Matrix')
plt.show()


## 2. Delay Predictor

Dataset: `Raahi_Behavioral_Chakachakit.csv`  
Task: regression for `predicted_delay`.

In [ ]:
behavioral_df = pd.read_csv(DATA_RAW_PATH / 'Raahi_Behavioral_Chakachakit.csv')
X_behavioral, y_behavioral, behavioral_encoders = preprocess_behavioral(behavioral_df)
X_train, X_test, y_train, y_test = train_test_split(
    X_behavioral, y_behavioral, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
delay_model = GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)
delay_model.fit(X_train, y_train)
delay_pred = delay_model.predict(X_test)
results['delay_predictor_r2'] = r2_score(y_test, delay_pred)
plt.figure(figsize=(6, 6))
plt.scatter(y_test, delay_pred, alpha=0.4)
plt.xlabel('Actual Delay')
plt.ylabel('Predicted Delay')
plt.title('Delay Predictor Regression Plot')
plt.grid(True, alpha=0.2)
plt.show()


## 3. Route Recommender

Dataset: `Raahi_Mumbai_Route_Transport_Noisy.csv`  
Task: binary classification for `is_recommended`.

In [ ]:
route_df = pd.read_csv(DATA_RAW_PATH / 'Raahi_Mumbai_Route_Transport_Noisy.csv')
X_route, y_route, route_encoders = preprocess_route(route_df)
X_train, X_test, y_train, y_test = train_test_split(
    X_route, y_route, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_route
)
route_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
route_model.fit(X_train, y_train)
route_pred = route_model.predict(X_test)
results['route_recommender_accuracy'] = accuracy_score(y_test, route_pred)
ConfusionMatrixDisplay.from_predictions(y_test, route_pred)
plt.title('Route Recommender Confusion Matrix')
plt.show()


## 4. Fare Model Reproduction Notes

Dataset: `Raahi_Full_Western_Line_Fares.csv`  
This section documents a reproducibility sketch for the existing fare models.

In [ ]:
fare_df = pd.read_csv(DATA_RAW_PATH / 'Raahi_Full_Western_Line_Fares.csv')
fare_features = fare_df.select_dtypes(include=['number']).copy()
fare_target = fare_features.iloc[:, -1]
fare_inputs = fare_features.iloc[:, :-1]
X_train, X_test, y_train, y_test = train_test_split(
    fare_inputs, fare_target, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
fare_model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
fare_model.fit(X_train, y_train)
fare_pred = fare_model.predict(X_test)
results['fare_model_r2_reproduction'] = r2_score(y_test, fare_pred)
plt.figure(figsize=(6, 6))
plt.scatter(y_test, fare_pred, alpha=0.4)
plt.xlabel('Actual Fare')
plt.ylabel('Predicted Fare')
plt.title('Fare Model Regression Plot')
plt.grid(True, alpha=0.2)
plt.show()


## 5. Emission Model Reproduction Notes

Dataset: `vehicle_emission_dataset.csv`  
This section documents a reproducibility sketch for the existing emission classifier.

In [ ]:
emission_df = pd.read_csv(DATA_RAW_PATH / 'vehicle_emission_dataset.csv').dropna().copy()
categorical_columns = ['Vehicle Type', 'Fuel Type', 'Road Type', 'Traffic Conditions']
encoders = {column: LabelEncoder().fit(emission_df[column].astype(str)) for column in categorical_columns}
for column, encoder in encoders.items():
    emission_df[f'{column}_enc'] = encoder.transform(emission_df[column].astype(str))
emission_target = LabelEncoder().fit_transform(pd.qcut(emission_df['CO2 Emissions'], q=3, labels=['low', 'medium', 'high']))
emission_features = emission_df[[f'{column}_enc' for column in categorical_columns] + ['Mileage', 'Speed']]
X_train, X_test, y_train, y_test = train_test_split(
    emission_features, emission_target, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=emission_target
)
emission_model = XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, eval_metric='mlogloss')
emission_model.fit(X_train, y_train)
emission_pred = emission_model.predict(X_test)
results['emission_model_accuracy_reproduction'] = accuracy_score(y_test, emission_pred)
ConfusionMatrixDisplay.from_predictions(y_test, emission_pred)
plt.title('Emission Model Confusion Matrix')
plt.show()


## Final Summary

In [ ]:
for metric_name, metric_value in results.items():
    print(f'{metric_name}: {metric_value:.4f}')
